In [5]:
# Preprocessing and CTNet Dataset Builder
import pandas as pd
import numpy as np
from scipy.signal import butter, filtfilt, resample
from sklearn.preprocessing import StandardScaler, LabelEncoder
import matplotlib.pyplot as plt

files = [
    'data/EEG_bar_trials_20_classes_3_20251118_164048.csv',
    'data/EEG_bar_trials_20_classes_3_20251118_174039.csv',
    'data/EEG_bar_trials_20_classes_3_20251118_174039.csv',
    'data/EEG_bar_trials_30_classes_3_20251118_161525.csv'
]
data = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)
print('Combined shape:', data.shape)

MI = data[data['phase']=='stimulus']
signals = MI.iloc[:,5:13].values
labels = MI['class_label'].values
splits = MI['split'].values
print('MI rows:', MI.shape)

def bandpass_filter(sig, low=4, high=40, fs=250, order=4):
    nyq = 0.5*fs
    b,a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b,a,sig)

fs = 250
filtered = np.zeros_like(signals)
for ch in range(8):
    filtered[:,ch] = bandpass_filter(signals[:,ch], fs=fs)

scaler = StandardScaler()
eeg_norm = scaler.fit_transform(filtered)

# ---- trial segmentation ----
trials = []
trial_labels = []
current = []
cur_label = labels[0]

for i in range(len(labels)):
    if labels[i] != cur_label:
        if len(current)>0:
            trials.append(np.array(current))
            trial_labels.append(cur_label)
        current=[]
        cur_label = labels[i]
    current.append(eeg_norm[i])

if len(current)>0:
    trials.append(np.array(current))
    trial_labels.append(cur_label)

print('Trials extracted:', len(trials))

target_len = fs*2  # 2 seconds
X=[]; y=[]
for tr, lbl in zip(trials, trial_labels):
    if len(tr)<50: continue
    tr_r = resample(tr, target_len)
    tr_r = (tr_r - tr_r.mean(0))/(tr_r.std(0)+1e-8)
    X.append(tr_r); y.append(lbl)

X = np.array(X)
y = np.array(y)
print('Resampled:', X.shape, y.shape)

X_ct = np.transpose(X,(0,2,1))
train_mask = np.isin(y, labels[splits=='train'])
test_mask = np.isin(y, labels[splits=='test'])
X_train = X_ct[train_mask]; y_train=y[train_mask]
X_test = X_ct[test_mask]; y_test=y[test_mask]
print('Train/Test:', X_train.shape, X_test.shape)

np.save('X_train_ctnet.npy', X_train)
np.save('y_train_ctnet.npy', y_train)
np.save('X_test_ctnet.npy', X_test)
np.save('y_test_ctnet.npy', y_test)
print('Saved!')


Combined shape: (614092, 22)
MI rows: (272274, 22)
Trials extracted: 192
Resampled: (192, 500, 8) (192,)
Train/Test: (192, 8, 500) (192, 8, 500)
Saved!
